# 04 · Reception Estimator (secondary model)

Predicts the probability that a review is **negative** (the 14% minority class), so the content team can prioritise
human audits. Features: reviewer history, playtime, price, platforms, days since release, and draft-time game features.
Game-level aggregates that *are* reception (positive ratio, rating, review counts) are excluded to avoid leakage.

* 5% uniform sample of the 41M reviews (~2M rows)
* Chronological split: train < 2022-01, validation 2022-H1, test >= 2022-07
* SageMaker built-in **XGBoost** training job, then **batch transform** on the test period, then **Model Registry**

In [ ]:
# Pin the SageMaker SDK v2 API used by this project (no-op if already installed)
%pip install -q "sagemaker>=2.232,<3"

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
def find_repo_root():
    """The notebooks import from src/, so the whole repo must be cloned, not just notebooks/."""
    candidates = [os.getcwd(), os.path.abspath(".."), os.path.expanduser("~/steam-store-copilot")]
    for path in candidates:
        if os.path.isdir(os.path.join(path, "src")) and os.path.exists(os.path.join(path, "src", "config.py")):
            return path
    raise RuntimeError("Could not find the repo's src/ folder. Clone the full repository in Studio "
                       "(Git > Clone Repository) and open the notebooks from steam-store-copilot/notebooks/.")


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

import boto3
import sagemaker

from src.config import S3Layout

sess = sagemaker.Session()
region = sess.boto_region_name
role = sagemaker.get_execution_role()  # LabRole in AWS Academy Learner Lab
layout = S3Layout(bucket=sess.default_bucket())
sm = boto3.client("sagemaker", region_name=region)
print("Region:", region)
print("Role:  ", role)
print("Bucket:", layout.bucket)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

from src.config import RECEPTION_MODEL_GROUP
from src.data.loaders import load_recs_games, load_store_games, load_users
from src.data.preprocess import build_tag_dataset
from src.data.s3io import fetch, read_text, stream, write_csv, write_text
from src.models.reception import features as rf

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})
reviews = rf.sample_reviews(stream(f"{layout.raw}/recs/recommendations.csv"), frac=0.05)
users = load_users(fetch(f"{layout.raw}/recs/users.csv"))
recs_games = load_recs_games(fetch(f"{layout.raw}/recs/games.csv"))
tag_ds = build_tag_dataset(load_store_games(fetch(f"{layout.raw}/store/games.csv")))
frame = rf.build_reception_frame(reviews, users, recs_games, tag_ds)
splits = rf.chronological_split(frame)
del reviews, users
pd.DataFrame({s: {"rows": len(d), "negative_share": d[rf.LABEL].mean(),
                  "from": d["date"].min().date(), "to": d["date"].max().date()} for s, d in splits.items()}).T

In [ ]:
prefix = layout.uri("reception")
write_csv(rf.to_xgboost_csv(splits["train"]), f"{prefix}/train/train.csv", index=False, header=False)
write_csv(rf.to_xgboost_csv(splits["val"]), f"{prefix}/validation/validation.csv", index=False, header=False)
write_csv(splits["test"][rf.FEATURES], f"{prefix}/test/test_features.csv", index=False, header=False)
print("Training data written under", prefix)

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

xgb_image = sagemaker.image_uris.retrieve("xgboost", region, version="1.7-1")
xgb = Estimator(
    image_uri=xgb_image, role=role, instance_count=1, instance_type="ml.m5.xlarge",
    output_path=f"{prefix}/models", base_job_name="reception-xgb", sagemaker_session=sess,
    hyperparameters={
        "objective": "binary:logistic", "eval_metric": "aucpr", "num_round": 300,
        "max_depth": 6, "eta": 0.1, "subsample": 0.8, "colsample_bytree": 0.8,
        "tree_method": "hist", "early_stopping_rounds": 20,
        "scale_pos_weight": round(rf.scale_pos_weight(splits["train"][rf.LABEL].values), 3),
    },
)
xgb.fit({"train": TrainingInput(f"{prefix}/train/", content_type="text/csv"),
         "validation": TrainingInput(f"{prefix}/validation/", content_type="text/csv")}, logs=False)
print("Training job:", xgb.latest_training_job.name)

## Batch transform on the held-out (most recent) period

In [ ]:
transformer = xgb.transformer(instance_count=1, instance_type="ml.m5.xlarge",
                              output_path=f"{prefix}/predictions", strategy="MultiRecord",
                              assemble_with="Line", accept="text/csv")
transformer.transform(f"{prefix}/test/test_features.csv", content_type="text/csv", split_type="Line",
                      wait=True, logs=False)
proba = np.array([float(x) for x in read_text(f"{prefix}/predictions/test_features.csv.out").split()])
y_test = splits["test"][rf.LABEL].values
metrics = {"pr_auc": average_precision_score(y_test, proba), "roc_auc": roc_auc_score(y_test, proba),
           "prevalence": float(y_test.mean()), "test_rows": int(len(y_test))}
metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
prec, rec, _ = precision_recall_curve(y_test, proba)
axes[0].plot(rec, prec, color="#1f4e8c", label=f"XGBoost (PR-AUC {metrics['pr_auc']:.3f})")
axes[0].axhline(metrics["prevalence"], color="grey", ls="--", label="random (prevalence)")
axes[0].set(xlabel="recall", ylabel="precision", title="Precision-recall, negative class")
axes[0].legend()
frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy="quantile")
axes[1].plot(mean_pred, frac_pos, marker="o", color="#1f4e8c", label="model")
axes[1].plot([0, 1], [0, 1], color="grey", ls="--", label="perfectly calibrated")
axes[1].set(xlabel="predicted P(negative)", ylabel="observed negative share", title="Calibration check")
axes[1].legend()
plt.tight_layout()
plt.show()
print("Note: scale_pos_weight inflates probabilities; use the score for ranking, not as a calibrated rate.")

In [ ]:
scored = splits["test"][["app_id"]].assign(p_negative=proba, negative=y_test)
priority = (scored.groupby("app_id").agg(reviews=("negative", "size"), predicted_negative=("p_negative", "mean"),
                                         actual_negative=("negative", "mean"))
            .query("reviews >= 50").sort_values("predicted_negative", ascending=False))
names = recs_games.set_index("app_id")["title"]
priority.insert(0, "title", names.reindex(priority.index).values)
print("Titles to prioritise for human audit (highest expected negative reception):")
priority.head(10).round(3)

## Register in the Model Registry

In [ ]:
from sagemaker.model_metrics import MetricsSource, ModelMetrics

metrics_uri = write_text(f"{prefix}/evaluation/evaluation.json", json.dumps(
    {"binary_classification_metrics": {"pr_auc": {"value": metrics["pr_auc"]},
                                       "roc_auc": {"value": metrics["roc_auc"]},
                                       "prevalence": {"value": metrics["prevalence"]}}}, indent=2))
package = xgb.register(
    content_types=["text/csv"], response_types=["text/csv"],
    inference_instances=["ml.m5.large"], transform_instances=["ml.m5.xlarge"],
    model_package_group_name=RECEPTION_MODEL_GROUP, approval_status="Approved",
    model_metrics=ModelMetrics(model_statistics=MetricsSource(s3_uri=metrics_uri, content_type="application/json")),
    description="Reception Estimator - XGBoost, P(negative review)",
)
print("Registered:", package.model_package_arn)